# Finetune T5-small baseline (BriefMe `arg_summ`)

**Reference baseline only** — [`briefme.train_t5_loop.run_t5_baseline_training`](../src/briefme/train_t5_loop.py), same as [`scripts/train_t5_baseline.py`](../scripts/train_t5_baseline.py).

**Order:** setup → **materialize** → **`t5_dataloaders_from_examples`** → **`run_t5_baseline_training`** (Trainer iterates only over the DataLoaders you pass) → optional **full run** (`RUN_FULL`). Data loading and training use **separate code cells**, each printing wall time so Hub/IO vs Trainer compute is obvious. **`logging_steps`** / **`eval_steps`** on `T5BaselineTrainConfig` drive step-granular **`log_history`** (see **`plot_t5_log_history`** in the setup cell). For scratch curves and experiment-tracking notes, see **[`04_train_scratch_seq2seq.ipynb`](04_train_scratch_seq2seq.ipynb)**.

**Outputs:** checkpoints under `output_dir`; **`summary.json`** after each completed run.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

from dotenv import load_dotenv


def _repo_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "src" / "briefme").is_dir():
        return cwd
    if (cwd.parent / "src" / "briefme").is_dir():
        return cwd.parent
    return cwd


REPO_ROOT = _repo_root()
SRC_ROOT = REPO_ROOT / "src"

try:
    import briefme as _briefme_check  # noqa: F401
except ImportError:
    if SRC_ROOT.is_dir():
        sys.path.insert(0, str(SRC_ROOT))

load_dotenv(REPO_ROOT / ".env", override=True)

import matplotlib.pyplot as plt


def plot_t5_log_history(log_history: list[dict], title: str = "T5 baseline") -> None:
    """Plot Trainer ``log_history``: train loss + eval ROUGE/F1 vs step (with ``logging_steps`` / ``eval_steps``)."""
    if not log_history:
        print("No log_history (empty run).")
        return
    train_x, train_y = [], []
    ev_x, rouge, f1 = [], [], []
    rt_x, rt_y = [], []
    for h in log_history:
        st = h.get("step")
        if st is None:
            continue
        if h.get("loss") is not None and h.get("eval_loss") is None:
            train_x.append(st)
            train_y.append(h["loss"])
        if h.get("eval_rougeL_f") is not None:
            ev_x.append(st)
            rouge.append(h["eval_rougeL_f"])
            f1.append(h.get("eval_token_f1_macro"))
        if h.get("eval_runtime") is not None:
            rt_x.append(st)
            rt_y.append(h["eval_runtime"])

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    ax0, ax1, ax2 = axes
    if train_x:
        ax0.plot(train_x, train_y, ".-", alpha=0.7)
    ax0.set_xlabel("step")
    ax0.set_ylabel("train loss")
    ax0.set_title("Training loss")
    ax0.grid(True, alpha=0.3)

    if ev_x:
        ax1.plot(ev_x, rouge, "o-", label="ROUGE-L F")
        ax1.plot(ev_x, f1, "s-", label="token F1 macro")
        ax1.legend(fontsize=8)
    ax1.set_xlabel("step")
    ax1.set_ylabel("0–1")
    ax1.set_title("Dev metrics (generate + aggregate)")
    ax1.grid(True, alpha=0.3)

    if rt_x:
        ax2.plot(rt_x, rt_y, "o-", color="steelblue")
        ax2.set_xlabel("step")
        ax2.set_ylabel("seconds")
        ax2.set_title("Eval runtime (Trainer)")
        ax2.grid(True, alpha=0.3)
    else:
        ax2.text(0.5, 0.5, "No eval_runtime in logs", ha="center", va="center")
        ax2.axis("off")

    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


print("REPO_ROOT:", REPO_ROOT)

### Sanity check

**Two cells:** **`[sanity — data pipeline]`** — load tokenizer and T5 weights, materialize from the Hub, build **DataLoaders** via `t5_dataloaders_from_examples`. **`[sanity — training]`** — call `run_t5_baseline_training` with `tokenizer`, `model`, `train_loader`, `eval_loader`, `data_collator` only. Re-run the training cell without re-streaming data. Small split + **1 epoch** to verify tokenizer, Trainer, and GPU path.

In [ ]:
import time
from pathlib import Path

from briefme.seq2seq_data import DEFAULT_T5_TASK_PREFIX, default_train_dev_materialize, get_t5_tokenizer
from briefme.train_t5_loop import T5BaselineTrainConfig, t5_dataloaders_from_examples
from transformer.config import HF_T5_BASELINE_MODEL_ID
from transformers import T5ForConditionalGeneration

SANITY_TRAIN_LIMIT = 128
SANITY_DEV_LIMIT = 32
SANITY_EPOCHS = 1
SANITY_BATCH = 4

SANITY_CFG = T5BaselineTrainConfig(
    model_id=HF_T5_BASELINE_MODEL_ID,
    output_dir=REPO_ROOT / "runs" / "notebook_t5_sanity",
    epochs=SANITY_EPOCHS,
    lr=3e-4,
    max_src_len=512,
    max_tgt_len=128,
    source_prefix=DEFAULT_T5_TASK_PREFIX,
    seed=42,
    logging_steps=8,
    eval_steps=16,
)

print("[briefme] loading tokenizer and T5 weights...", flush=True)
_t_sanity_data0 = time.perf_counter()
tokenizer = get_t5_tokenizer(SANITY_CFG.model_id)
model = T5ForConditionalGeneration.from_pretrained(SANITY_CFG.model_id)
train_ex, dev_ex = default_train_dev_materialize(SANITY_TRAIN_LIMIT, SANITY_DEV_LIMIT)
sanity_train_loader, sanity_eval_loader, sanity_collator = t5_dataloaders_from_examples(
    train_ex,
    dev_ex,
    tokenizer,
    model,
    SANITY_CFG,
    batch_size=SANITY_BATCH,
)
print(f"[sanity — data pipeline] wall time: {time.perf_counter() - _t_sanity_data0:.2f}s", flush=True)

In [ ]:
import time

from briefme.train_t5_loop import run_t5_baseline_training

_t_sanity_train0 = time.perf_counter()
sanity_summary = run_t5_baseline_training(
    SANITY_CFG,
    tokenizer=tokenizer,
    model=model,
    train_loader=sanity_train_loader,
    eval_loader=sanity_eval_loader,
    data_collator=sanity_collator,
)
print(f"[sanity — training] wall time: {time.perf_counter() - _t_sanity_train0:.2f}s", flush=True)
plot_t5_log_history(sanity_summary.get("log_history", []), title="T5 sanity")
sanity_summary

### Full training (optional)

Same split as sanity: **`[full — data pipeline]`** (tokenizer, model, materialize, dataloaders) vs **`[full — training]`** (`run_t5_baseline_training` only). Each cell prints its own wall time.

**Curves:** `FULL_CFG` uses **`logging_steps`** / **`eval_steps`** (set **`eval_steps=None`** for once-per-epoch eval only). **`summary.json`** includes **`log_history`** for plotting.

Set **`RUN_FULL = True`** after sanity looks good. Use **`train_limit=None`** in `default_train_dev_materialize` (or `--train-limit none` on the CLI) for the full Hub train split. Large runs are often easier from the **terminal** (`scripts/train_t5_baseline.py`).

In [ ]:
from pathlib import Path

from briefme.seq2seq_data import DEFAULT_T5_TASK_PREFIX
from briefme.train_t5_loop import T5BaselineTrainConfig
from transformer.config import HF_T5_BASELINE_MODEL_ID

FULL_CFG = T5BaselineTrainConfig(
    model_id=HF_T5_BASELINE_MODEL_ID,
    output_dir=REPO_ROOT / "runs" / "notebook_t5_baseline",
    epochs=2,
    lr=3e-4,
    max_src_len=512,
    max_tgt_len=128,
    source_prefix=DEFAULT_T5_TASK_PREFIX,
    seed=42,
    logging_steps=50,
    eval_steps=256,
)

FULL_TRAIN_LIMIT = 4096
FULL_DEV_LIMIT = 512
FULL_BATCH = 8

In [ ]:
import time

from briefme.seq2seq_data import default_train_dev_materialize, get_t5_tokenizer
from briefme.train_t5_loop import t5_dataloaders_from_examples
from transformers import T5ForConditionalGeneration

RUN_FULL = False

if RUN_FULL:
    _t_full_data0 = time.perf_counter()
    tokenizer_f = get_t5_tokenizer(FULL_CFG.model_id)
    model_f = T5ForConditionalGeneration.from_pretrained(FULL_CFG.model_id)
    full_train_ex, full_dev_ex = default_train_dev_materialize(FULL_TRAIN_LIMIT, FULL_DEV_LIMIT)
    full_train_loader, full_eval_loader, full_collator = t5_dataloaders_from_examples(
        full_train_ex,
        full_dev_ex,
        tokenizer_f,
        model_f,
        FULL_CFG,
        batch_size=FULL_BATCH,
    )
    print(f"[full — data pipeline] wall time: {time.perf_counter() - _t_full_data0:.2f}s", flush=True)

In [ ]:
import time

from briefme.train_t5_loop import run_t5_baseline_training

if RUN_FULL:
    _t_full_train0 = time.perf_counter()
    full_summary = run_t5_baseline_training(
        FULL_CFG,
        tokenizer=tokenizer_f,
        model=model_f,
        train_loader=full_train_loader,
        eval_loader=full_eval_loader,
        data_collator=full_collator,
    )
    print(f"[full — training] wall time: {time.perf_counter() - _t_full_train0:.2f}s", flush=True)
    plot_t5_log_history(full_summary.get("log_history", []), title="T5 full notebook run")
    print(full_summary)